In [1]:
print(1, 2, 3, 4)


1 2 3 4


In [2]:
from dotenv import load_dotenv
load_dotenv()                # → True

from openai import OpenAI
openai_client = OpenAI()     # no error = done
MODEL = "gpt-5.4-mini" 

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
# inspect structure 
print("keys:", list(documents[0].keys()))

doc = documents[0]
preview = {
    k: (v[:120] + " …[truncated]" if isinstance(v, str) and len(v) > 120 else v)
    for k, v in doc.items()
}
import json
print(json.dumps(preview, indent=4, ensure_ascii=False))

keys: ['content', 'filename']
{
    "content": "# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8 …[truncated]",
    "filename": "01-agentic-rag/lessons/01-intro.md"
}


In [6]:
#Q1 Number of documents
print(len(documents))

72


In [7]:
#Q2 What's the first name of the search 'How does the agentic loop keep calling the model until it stops?'

from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

for r in results:
    print(r["filename"])

print("\nANSWER:", results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md

ANSWER: 01-agentic-rag/lessons/14-agentic-loop.md


In [8]:
#Q3 — download the helper
import requests

url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py"
with open("rag_helper.py", "w") as f:
    f.write(requests.get(url).text)

print("saved rag_helper.py")

saved rag_helper.py


In [9]:
#Q3 RAGBase helper to filename/content schema
from rag_helper import RAGBase

class LessonRAG(RAGBase):
    # override 1: plain search on own index (drop FAQ course-filter / question-boost)
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    # override 2 build context from filename/content, not section/question/answer
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"FILE: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    # override 3 return the WHOLE response (base returns only .output_text)
    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt},
        ]
        return self.llm_client.responses.create(model=self.model, input=input_messages)

    # override 4 return (answer, usage) instead of just the answer
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage

In [10]:
rag = LessonRAG(index=index, llm_client=openai_client, model=MODEL)

answer, usage = rag.rag("How does the agentic loop keep calling the model until it stops?")

print(answer)
print("\ninput tokens:", usage.input_tokens)

The loop keeps calling the model by checking whether the response contains any `function_call` items.

- It sends the current `messages` to the model.
- If the model returns a function call, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- If there are no function calls, `has_function_calls` stays `False`, and the loop `break`s.

So the stop condition is: **no function calls in the model response**.

input tokens: 7126


In [11]:
#Q4 chunking
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print("chunks:", len(chunks))

chunks: 295


In [13]:
#Q5 RAG with chunks
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

rag_chunked = LessonRAG(index=chunk_index, llm_client=openai_client, model=MODEL)
answerRag, usageRag = rag_chunked.rag("How does the agentic loop keep calling the model until it stops?")

print(answerRag)
print(usageRag.input_tokens)
print("\nchunked input tokens:", usageRag.input_tokens)
print("fewer tokens than Q3:", 7126 - usageRag.input_tokens)

The loop keeps calling the model with a `while True` loop. After each call, it checks whether any `function_call` items were returned:

- If there are function calls, it runs them, appends the results to `messages`, and continues.
- If there are no function calls in that turn, it `break`s out of the loop.

So the stop condition is: **no function calls this turn means the model has produced its final answer, and the loop ends.**
2309

chunked input tokens: 2309
fewer tokens than Q3: 4817


In [14]:
#Q6 Install toyaikit 
!uv add toyaikit

Resolved 128 packages in 973ms                                       
Prepared 6 packages in 369ms                                             
Installed 7 packages in 22ms                                
 + anthropic==0.111.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.66
 + httpcore2==2.4.0
 + httpx2==2.4.0
 + toyaikit==0.0.11
 + truststore==0.10.4


In [15]:
search_calls = 0

def search(query: str) -> list:
    """Search the course materials for content relevant to the query.

    Args:
        query: what to look for in the course materials
    """
    global search_calls
    search_calls += 1
    print(f"  [search #{search_calls}] {query!r}")
    return chunk_index.search(query, num_results=5)

In [16]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner

tools = Tools()
tools.add_tool(search)

agent_client = OpenAIClient(model=MODEL, client=openai_client)

instructions = (
    "You're a course teaching assistant. Answer the student's question "
    "using the search tool. Make multiple searches with different keywords "
    "before answering."
)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=instructions,
    llm_client=agent_client,
)

search_calls = 0   # reset right before the run
result = runner.loop(prompt="How does the agentic loop work, and how is it different from plain RAG?")

print("\n>>> search called:", search_calls, "times")

  [search #1] 'agentic loop RAG course materials'
  [search #2] 'plain RAG retrieval augmented generation course materials'
  [search #3] 'agentic loop tool use reasoning planning course materials'

>>> search called: 3 times
